# Bronze Data Inspection

This notebook is only for early inspection of the generated Bronze layer.
It creates a Spark session, loads the three raw parquet datasets, and shows schema and sample rows.

## What To Check

- Bronze files exist under `data/bronze/`
- Raw schemas match the assignment inputs
- Data quality issues are visible before Silver cleaning
- Row counts are large enough for the rest of the assignment

In [ ]:
import os
from pathlib import Path

for env_name in ("SPARK_HOME", "HADOOP_CONF_DIR", "SPARK_CONF_DIR"):
    os.environ.pop(env_name, None)

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window


def standardized_timestamp_string(column_name, slash_pattern="dd/MM/yyyy HH:mm"):
    value = F.trim(F.col(column_name))
    standard_ts = F.try_to_timestamp(value, F.lit("yyyy-MM-dd HH:mm:ss"))
    slash_ts = F.try_to_timestamp(value, F.lit(slash_pattern))
    return (
        F.when(
            standard_ts.isNotNull(),
            F.date_format(standard_ts, "yyyy-MM-dd HH:mm:ss"),
        )
        .when(
            slash_ts.isNotNull(),
            F.date_format(slash_ts, "yyyy-MM-dd HH:mm:ss"),
        )
        .otherwise(F.lit(None).cast("string"))
    )


def format_then_cast_timestamp(column_name, slash_pattern="dd/MM/yyyy HH:mm"):
    return standardized_timestamp_string(
        column_name,
        slash_pattern=slash_pattern,
    ).cast("timestamp")


In [2]:
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

bronze_root = project_root / "data" / "edtech_raw_datasets_3_files"

leads_path = (bronze_root / "leads_raw.csv").resolve().as_uri()
learning_path = (bronze_root / "learning_activity_raw.csv").resolve().as_uri()
sales_path = (bronze_root / "sales_raw.csv").resolve().as_uri()

print("Project root:", project_root)
print("Bronze root:", bronze_root)
print("Leads path:", leads_path)
print("Learning path:", learning_path)
print("Sales path:", sales_path)

Project root: /mnt/c/Users/nadda/Desktop/work/DE-assignment-learn-corporation
Bronze root: /mnt/c/Users/nadda/Desktop/work/DE-assignment-learn-corporation/data/edtech_raw_datasets_3_files
Leads path: file:///mnt/c/Users/nadda/Desktop/work/DE-assignment-learn-corporation/data/edtech_raw_datasets_3_files/leads_raw.csv
Learning path: file:///mnt/c/Users/nadda/Desktop/work/DE-assignment-learn-corporation/data/edtech_raw_datasets_3_files/learning_activity_raw.csv
Sales path: file:///mnt/c/Users/nadda/Desktop/work/DE-assignment-learn-corporation/data/edtech_raw_datasets_3_files/sales_raw.csv


In [3]:
spark = (
    SparkSession.builder
    .appName("learn-bronze-inspection")
    .master("local[*]")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.hadoop.fs.defaultFS", "file:///")
    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/24 23:20:03 WARN Utils: Your hostname, DESKTOP-8HR4EC1, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/24 23:20:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 23:20:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
leads_df = (
    spark.read
    .option("header", True)
    .csv(str(leads_path))
)

learning_df = (
    spark.read
    .option("header", True)
    .csv(str(learning_path))
)

sales_df = (
    spark.read
    .option("header", True)
    .csv(str(sales_path))
)

In [5]:
row_counts = [
    ("leads", leads_df.count()),
    ("learning_activity", learning_df.count()),
    ("sales", sales_df.count()),
]

spark.createDataFrame(row_counts, ["dataset", "row_count"]).show(truncate=False)

+-----------------+---------+
|dataset          |row_count|
+-----------------+---------+
|leads            |1000     |
|learning_activity|1000     |
|sales            |1000     |
+-----------------+---------+



## Bronze Schemas

In [6]:
print("Leads schema")
leads_df.printSchema()

print("Learning schema")
learning_df.printSchema()

print("Sales schema")
sales_df.printSchema()

Leads schema
root
 |-- student_code: string (nullable = true)
 |-- student_name: string (nullable = true)
 |-- mobile_phone: string (nullable = true)
 |-- school_name: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- lead_created_at: string (nullable = true)
 |-- source_channel: string (nullable = true)

Learning schema
root
 |-- student_code: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- event_name: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- chapter: string (nullable = true)
 |-- duration_minutes: string (nullable = true)
 |-- platform: string (nullable = true)

Sales schema
root
 |-- student_code: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- payment_status: string (nullable = true)



## Sample Bronze Rows

In [7]:
leads_df.show(100, truncate=False)

+------------+----------------+------------+----------------------------+-----+-------------------+--------------+
|student_code|student_name    |mobile_phone|school_name                 |grade|lead_created_at    |source_channel|
+------------+----------------+------------+----------------------------+-----+-------------------+--------------+
|STU100302   |Nina Pramoj     |0070012089  |St. Gabriel's College       |M4   |2025-12-22 07:11:00|line oa       |
|STU100293   |Kwan Siripong   |0752772693  |Montfort College            |M3   |2025-02-19 14:11:00|school fair   |
|STU100350   |Ploy Viroj      |0335843230  |Bangkok Christian College   |M3   |2025-12-05 01:05:00|tiktok        |
|STU100386   |Leo Ong         |035-215-8837|Debsirin School             |M2   |2025-09-30 09:24:00|LINE          |
|NULL        |Leo Pramoj      |0895951226  |Samsen Wittayalai School    |M3   |2025-08-11 22:58:00|Facebook Ads  |
|STU100261   |Ice Pramoj      |0769243075  |Satriwithaya School         |M3   |2

In [8]:
learning_df.show(100, truncate=False)

+------------+-------------------+---------------+--------------+------------+----------------+--------+
|student_code|event_time         |event_name     |subject       |chapter     |duration_minutes|platform|
+------------+-------------------+---------------+--------------+------------+----------------+--------+
|STU100294   |2026-02-27 16:58:00|download_sheet |English       |Chapter 2   |5               |web     |
|STU100238   |2025-07-02 21:44:00|download_sheet |Physics       |Chapter 3   |-46             |android |
|STU100099   |2026-02-25 08:37:00|video_watch    |Math          |Chapter 1   |43              |ios     |
|STU100238   |2025-08-03 17:03:00|quiz_submit    |Physics       |Chapter 2   |80              |tablet  |
|STU100020   |2025-12-26 01:46:00|practice_finish|Biology       |Chapter 1   |12              |tablet  |
|STU100398   |03/10/2026 23:55   |video_watch    |Social Studies|Intro       |70              |ios     |
|STU100281   |2025-08-05 19:33:00|practice_start |Thai 

In [9]:
sales_df.show(100, truncate=False)

+------------+------------+-------------------+---------------------------+---------------+-------+--------------+
|student_code|order_id    |order_date         |product_name               |product_type   |amount |payment_status|
+------------+------------+-------------------+---------------------------+---------------+-------+--------------+
|STU100334   |ORD202500000|2025-10-19 03:06:00|Chemistry Crash Course     |recorded_course|4990   |Complete      |
|STU100407   |ORD202500000|2025-10-01 12:15:00|Math Intensive Bootcamp    |mock_exam_pack |990    |completed     |
|STU100286   |ORD202500002|2025-10-28 07:58:00|Chemistry Crash Course     |trial_course   |0.99   |failed        |
|STU100103   |ORD202500003|2025-12-10 14:14:00|M6 Final Review Bundle     |recorded_course|1990   |completed     |
|STU100003   |ORD202500002|2026-02-02 00:35:00|English Grammar Accelerator|live_course    |4990   |Complete      |
|STU100366   |ORD202500005|2025-12-04 22:27:00|Biology Exam Prep          |bundl

## Quick Raw Data Quality Checks

These are not Silver transformations yet. They are only inspection queries to confirm the synthetic Bronze layer contains the expected issues.

In [10]:
leads_df.select(
    F.count(F.when(F.col("student_code").isNull() | (F.trim(F.col("student_code")) == ""), 1)).alias("missing_student_code_rows"),
    F.count("*").alias("total_rows")
).show()

+-------------------------+----------+
|missing_student_code_rows|total_rows|
+-------------------------+----------+
|                       11|      1000|
+-------------------------+----------+



In [11]:
learning_df.select(
    F.count(F.when(F.col("student_code").isNull() | (F.trim(F.col("student_code")) == ""), 1)).alias("missing_student_code_rows"),
    F.count(F.when(F.col("event_time").isNull(), 1)).alias("missing_event_time_rows"),
    F.count(F.when(F.col("duration_minutes") < 0, 1)).alias("negative_duration_rows")
).show()

+-------------------------+-----------------------+----------------------+
|missing_student_code_rows|missing_event_time_rows|negative_duration_rows|
+-------------------------+-----------------------+----------------------+
|                       22|                     21|                    32|
+-------------------------+-----------------------+----------------------+



In [12]:
sales_df.groupBy("order_id").count().filter(F.col("count") > 1).count()

47

## Next Step

Once this notebook looks correct, the next notebook section should implement `leads_silver`, `learning_silver`, and `sales_silver` as pure PySpark transformations.

# Transform to silver

## Leads

In [13]:
bronze_leads_df = leads_df
print(f"Records = {bronze_leads_df.count()}")

bronze_leads_df.show()

Records = 1000
+------------+--------------+------------+--------------------+-----+-------------------+--------------+
|student_code|  student_name|mobile_phone|         school_name|grade|    lead_created_at|source_channel|
+------------+--------------+------------+--------------------+-----+-------------------+--------------+
|   STU100302|   Nina Pramoj|  0070012089|St. Gabriel's Col...|   M4|2025-12-22 07:11:00|       line oa|
|   STU100293| Kwan Siripong|  0752772693|    Montfort College|   M3|2025-02-19 14:11:00|   school fair|
|   STU100350|    Ploy Viroj|  0335843230|Bangkok Christian...|   M3|2025-12-05 01:05:00|        tiktok|
|   STU100386|       Leo Ong|035-215-8837|     Debsirin School|   M2|2025-09-30 09:24:00|          LINE|
|        NULL|    Leo Pramoj|  0895951226|Samsen Wittayalai...|   M3|2025-08-11 22:58:00|  Facebook Ads|
|   STU100261|    Ice Pramoj|  0769243075| Satriwithaya School|   M3|2025-10-09 01:00:00|       organic|
|   STU100144|Napat Siripong|  009221346

### Trim student code space

In [14]:
bronze_leads_df = bronze_leads_df.withColumn(
    "student_code",
    F.trim(F.col("student_code"))
)

bronze_leads_df.show()

+------------+--------------+------------+--------------------+-----+-------------------+--------------+
|student_code|  student_name|mobile_phone|         school_name|grade|    lead_created_at|source_channel|
+------------+--------------+------------+--------------------+-----+-------------------+--------------+
|   STU100302|   Nina Pramoj|  0070012089|St. Gabriel's Col...|   M4|2025-12-22 07:11:00|       line oa|
|   STU100293| Kwan Siripong|  0752772693|    Montfort College|   M3|2025-02-19 14:11:00|   school fair|
|   STU100350|    Ploy Viroj|  0335843230|Bangkok Christian...|   M3|2025-12-05 01:05:00|        tiktok|
|   STU100386|       Leo Ong|035-215-8837|     Debsirin School|   M2|2025-09-30 09:24:00|          LINE|
|        NULL|    Leo Pramoj|  0895951226|Samsen Wittayalai...|   M3|2025-08-11 22:58:00|  Facebook Ads|
|   STU100261|    Ice Pramoj|  0769243075| Satriwithaya School|   M3|2025-10-09 01:00:00|       organic|
|   STU100144|Napat Siripong|  0092213461|     Debsirin

### Normalize Mobile Phone

In [15]:
phone_digits = F.regexp_replace(
    F.coalesce(F.col("mobile_phone"), F.lit("")),
    r"[^0-9]",
    ""
)

bronze_leads_df = (
    bronze_leads_df
    .withColumn("phone_digits", phone_digits)
    .withColumn("mobile_phone",
        F.when(
            F.col("phone_digits").rlike(r"^66\d{9}$"),
            F.concat(F.lit("0"), F.substring(F.col("phone_digits"), 3, 9))
        )
        .when(
            F.col("phone_digits").rlike(r"^0\d{9}$"),
            F.col("phone_digits")
        )
        .otherwise(F.lit(None).cast("string"))
    )
    .drop("phone_digits")
)


bronze_leads_df.show()

+------------+--------------+------------+--------------------+-----+-------------------+--------------+
|student_code|  student_name|mobile_phone|         school_name|grade|    lead_created_at|source_channel|
+------------+--------------+------------+--------------------+-----+-------------------+--------------+
|   STU100302|   Nina Pramoj|  0070012089|St. Gabriel's Col...|   M4|2025-12-22 07:11:00|       line oa|
|   STU100293| Kwan Siripong|  0752772693|    Montfort College|   M3|2025-02-19 14:11:00|   school fair|
|   STU100350|    Ploy Viroj|  0335843230|Bangkok Christian...|   M3|2025-12-05 01:05:00|        tiktok|
|   STU100386|       Leo Ong|  0352158837|     Debsirin School|   M2|2025-09-30 09:24:00|          LINE|
|        NULL|    Leo Pramoj|  0895951226|Samsen Wittayalai...|   M3|2025-08-11 22:58:00|  Facebook Ads|
|   STU100261|    Ice Pramoj|  0769243075| Satriwithaya School|   M3|2025-10-09 01:00:00|       organic|
|   STU100144|Napat Siripong|  0092213461|     Debsirin

### Normalize source_channel

In [16]:
bronze_leads_df.select(F.col("source_channel")).distinct().show(50)

+--------------+
|source_channel|
+--------------+
|       WEBINAR|
| GOOGLE SEARCH|
|       Organic|
|          Line|
|      REFERRAL|
|   school fair|
|       ORGANIC|
| Google Search|
|        TikTok|
|          LINE|
|      referral|
|      FACEBOOK|
|        tiktok|
|       Webinar|
|       Line Oa|
|       organic|
|      Referral|
|       webinar|
|  Facebook Ads|
|        Tiktok|
|        TIKTOK|
|        Google|
|   SCHOOL FAIR|
|       LINE OA|
|        GOOGLE|
|  FACEBOOK ADS|
|   School Fair|
|      facebook|
|      Facebook|
|       line oa|
|        google|
+--------------+



In [17]:
bronze_leads_df = bronze_leads_df.withColumn(
    "lowercase_source_channel", F.lower(F.trim(F.col("source_channel")))
).withColumn(
    "source_channel_clean",
    F.when(
        F.col("lowercase_source_channel").contains("facebook"),
        F.lit("facebook")
    )
    .when(
        F.col("lowercase_source_channel").contains("google"),
        F.lit("google")
    )
    .when(
        F.col("lowercase_source_channel").contains("line"),
        F.lit("line")
    )
    .when(
        F.col("lowercase_source_channel").contains("webinar"),
        F.lit("webinar")
    )
    .when(
        F.col("lowercase_source_channel").contains("referral"),
        F.lit("referral")
    )
    .when(
        F.col("lowercase_source_channel").contains("tiktok"),
        F.lit("tiktok")
    )
    .when(
        F.col("lowercase_source_channel").contains("organic"),
        F.lit("organic")
    )
    .when(
        F.col("lowercase_source_channel").contains("school fair"),
        F.lit("school_fair")
    )
).drop(
    "source_channel",
    "lowercase_source_channel"
).withColumn(
    "source_channel", F.col("source_channel_clean")
).drop(
    "source_channel_clean"
)

bronze_leads_df.show(truncate=False)

+------------+--------------+------------+----------------------------+-----+-------------------+--------------+
|student_code|student_name  |mobile_phone|school_name                 |grade|lead_created_at    |source_channel|
+------------+--------------+------------+----------------------------+-----+-------------------+--------------+
|STU100302   |Nina Pramoj   |0070012089  |St. Gabriel's College       |M4   |2025-12-22 07:11:00|line          |
|STU100293   |Kwan Siripong |0752772693  |Montfort College            |M3   |2025-02-19 14:11:00|school_fair   |
|STU100350   |Ploy Viroj    |0335843230  |Bangkok Christian College   |M3   |2025-12-05 01:05:00|tiktok        |
|STU100386   |Leo Ong       |0352158837  |Debsirin School             |M2   |2025-09-30 09:24:00|line          |
|NULL        |Leo Pramoj    |0895951226  |Samsen Wittayalai School    |M3   |2025-08-11 22:58:00|facebook      |
|STU100261   |Ice Pramoj    |0769243075  |Satriwithaya School         |M3   |2025-10-09 01:00:00

### Normalize timestamp

In [18]:
bronze_leads_df = (
    bronze_leads_df
    .withColumn(
        "lead_created_at",
        format_then_cast_timestamp("lead_created_at")
    )
)

bronze_leads_df.show(50)

+------------+--------------+------------+--------------------+-----+--------------+-------------------+
|student_code|  student_name|mobile_phone|         school_name|grade|source_channel|    lead_created_at|
+------------+--------------+------------+--------------------+-----+--------------+-------------------+
|   STU100302|   Nina Pramoj|  0070012089|St. Gabriel's Col...|   M4|          line|2025-12-22 07:11:00|
|   STU100293| Kwan Siripong|  0752772693|    Montfort College|   M3|   school_fair|2025-02-19 14:11:00|
|   STU100350|    Ploy Viroj|  0335843230|Bangkok Christian...|   M3|        tiktok|2025-12-05 01:05:00|
|   STU100386|       Leo Ong|  0352158837|     Debsirin School|   M2|          line|2025-09-30 09:24:00|
|        NULL|    Leo Pramoj|  0895951226|Samsen Wittayalai...|   M3|      facebook|2025-08-11 22:58:00|
|   STU100261|    Ice Pramoj|  0769243075| Satriwithaya School|   M3|       organic|2025-10-09 01:00:00|
|   STU100144|Napat Siripong|  0092213461|     Debsirin

In [19]:
bronze_leads_df.printSchema()

root
 |-- student_code: string (nullable = true)
 |-- student_name: string (nullable = true)
 |-- mobile_phone: string (nullable = true)
 |-- school_name: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- source_channel: string (nullable = true)
 |-- lead_created_at: timestamp (nullable = true)



### Deduplicate and drop NULL value on student_code

In [20]:
bronze_leads_df.filter(
    F.col("student_code").isNotNull()
).select(
    "student_code",
    "student_name",
    "lead_created_at"
).orderBy("student_code").show(2000,truncate=False)

+------------+----------------+-------------------+
|student_code|student_name    |lead_created_at    |
+------------+----------------+-------------------+
|STU100000   |Petch Lertchai  |2025-01-01 09:28:00|
|STU100001   |Jane Srisuk     |2025-10-12 22:45:00|
|STU100001   |Jane Srisuk     |2025-12-23 02:36:00|
|STU100001   |Jane Srisuk     |2025-10-09 18:52:00|
|STU100001   |Jane Srisuk     |2025-05-19 05:07:00|
|STU100001   |Jane Srisuk     |2025-06-23 13:47:00|
|STU100001   |Jane Srisuk     |2026-01-19 14:33:00|
|STU100002   |Arisa Rattan    |2025-11-13 11:55:00|
|STU100002   |Arisa Rattan    |2025-12-13 03:56:00|
|STU100002   |Arisa Rattan    |2025-06-19 03:03:00|
|STU100002   |Arisa Rattan    |2025-04-07 15:35:00|
|STU100003   |Leo Ong         |2025-09-19 12:10:00|
|STU100004   |Petch Khan      |2025-02-16 07:19:00|
|STU100004   |Petch Khan      |2025-04-23 15:43:00|
|STU100005   |Petch Viroj     |2026-02-05 03:11:00|
|STU100005   |Petch Viroj     |2025-08-04 02:23:00|
|STU100006  

In [21]:
latest_lead_window = Window.partitionBy("student_code").orderBy(
    F.col("lead_created_at").desc_nulls_last()
)

bronze_leads_df = (
    bronze_leads_df
    .filter(
        (F.col("student_code").rlike(r"^STU\d{6}$"))
    )
    .withColumn(
        "rn", F.row_number().over(latest_lead_window)
    )
    .filter(
        F.col("rn") == 1
    )
    .drop(
        "rn"
    )
)

bronze_leads_df.show(truncate=False)

+------------+----------------+------------+----------------------------+-----+--------------+-------------------+
|student_code|student_name    |mobile_phone|school_name                 |grade|source_channel|lead_created_at    |
+------------+----------------+------------+----------------------------+-----+--------------+-------------------+
|STU100000   |Petch Lertchai  |0022768040  |Satriwithaya School         |M2   |school_fair   |2025-01-01 09:28:00|
|STU100001   |Jane Srisuk     |0577442871  |St. Gabriel's College       |M3   |tiktok        |2026-01-19 14:33:00|
|STU100002   |Arisa Rattan    |0930086875  |St. Gabriel's College       |M1   |school_fair   |2025-12-13 03:56:00|
|STU100003   |Leo Ong         |0831067988  |St. Gabriel's College       |M3   |line          |2025-09-19 12:10:00|
|STU100004   |Petch Khan      |0136760652  |Samsen Wittayalai School    |M4   |tiktok        |2025-04-23 15:43:00|
|STU100005   |Petch Viroj     |0237088409  |Horwang School              |M6   |t

### Silver Schema

In [22]:
silver_leads_df = bronze_leads_df.select(
    "student_code",
    "student_name",
    "mobile_phone",
    "school_name",
    "grade",
    "lead_created_at",
    "source_channel"
)

print(silver_leads_df.count())
silver_leads_df.printSchema()

375
root
 |-- student_code: string (nullable = true)
 |-- student_name: string (nullable = true)
 |-- mobile_phone: string (nullable = true)
 |-- school_name: string (nullable = true)
 |-- grade: string (nullable = true)
 |-- lead_created_at: timestamp (nullable = true)
 |-- source_channel: string (nullable = true)



In [43]:
silver_leads_df.show(1000,truncate=False)

+------------+----------------+------------+----------------------------+-----+-------------------+--------------+
|student_code|student_name    |mobile_phone|school_name                 |grade|lead_created_at    |source_channel|
+------------+----------------+------------+----------------------------+-----+-------------------+--------------+
|STU100000   |Petch Lertchai  |0022768040  |Satriwithaya School         |M2   |2025-01-01 09:28:00|school_fair   |
|STU100001   |Jane Srisuk     |0577442871  |St. Gabriel's College       |M3   |2026-01-19 14:33:00|tiktok        |
|STU100002   |Arisa Rattan    |0930086875  |St. Gabriel's College       |M1   |2025-12-13 03:56:00|school_fair   |
|STU100003   |Leo Ong         |0831067988  |St. Gabriel's College       |M3   |2025-09-19 12:10:00|line          |
|STU100004   |Petch Khan      |0136760652  |Samsen Wittayalai School    |M4   |2025-04-23 15:43:00|tiktok        |
|STU100005   |Petch Viroj     |0237088409  |Horwang School              |M6   |2

## Learning

In [23]:
bronze_learning_df = learning_df
print(f"Records = {bronze_learning_df.count()}")

bronze_learning_df.show()

Records = 1000
+------------+-------------------+---------------+--------------+------------+----------------+--------+
|student_code|         event_time|     event_name|       subject|     chapter|duration_minutes|platform|
+------------+-------------------+---------------+--------------+------------+----------------+--------+
|   STU100294|2026-02-27 16:58:00| download_sheet|       English|   Chapter 2|               5|     web|
|   STU100238|2025-07-02 21:44:00| download_sheet|       Physics|   Chapter 3|             -46| android|
|   STU100099|2026-02-25 08:37:00|    video_watch|          Math|   Chapter 1|              43|     ios|
|   STU100238|2025-08-03 17:03:00|    quiz_submit|       Physics|   Chapter 2|              80|  tablet|
|   STU100020|2025-12-26 01:46:00|practice_finish|       Biology|   Chapter 1|              12|  tablet|
|   STU100398|   03/10/2026 23:55|    video_watch|Social Studies|       Intro|              70|     ios|
|   STU100281|2025-08-05 19:33:00| pract

### Trim student code

In [24]:
bronze_learning_df = bronze_learning_df.withColumn(
    "student_code",
    F.trim(F.col("student_code"))
)

bronze_learning_df.show()

+------------+-------------------+---------------+--------------+------------+----------------+--------+
|student_code|         event_time|     event_name|       subject|     chapter|duration_minutes|platform|
+------------+-------------------+---------------+--------------+------------+----------------+--------+
|   STU100294|2026-02-27 16:58:00| download_sheet|       English|   Chapter 2|               5|     web|
|   STU100238|2025-07-02 21:44:00| download_sheet|       Physics|   Chapter 3|             -46| android|
|   STU100099|2026-02-25 08:37:00|    video_watch|          Math|   Chapter 1|              43|     ios|
|   STU100238|2025-08-03 17:03:00|    quiz_submit|       Physics|   Chapter 2|              80|  tablet|
|   STU100020|2025-12-26 01:46:00|practice_finish|       Biology|   Chapter 1|              12|  tablet|
|   STU100398|   03/10/2026 23:55|    video_watch|Social Studies|       Intro|              70|     ios|
|   STU100281|2025-08-05 19:33:00| practice_start|     

### Drop NULL on student_code and event_time, and negative duration_minutes

In [25]:
bronze_learning_df = (
        bronze_learning_df
        .filter(
            (F.col("student_code").rlike(r"^STU\d{6}$")) &
            (F.col("event_time").isNotNull()) &
            (F.col("duration_minutes") >= 0)
        )
        .withColumn(
            "duration_minutes_int", F.col("duration_minutes").cast("int")
        )
        .drop(
            "duration_minutes"
        )
        .withColumn(
            "duration_minutes", F.col("duration_minutes_int")
        )
        .drop(
            "duration_minutes_int"
        )
)

bronze_learning_df.show()

+------------+-------------------+---------------+--------------+------------+--------+----------------+
|student_code|         event_time|     event_name|       subject|     chapter|platform|duration_minutes|
+------------+-------------------+---------------+--------------+------------+--------+----------------+
|   STU100294|2026-02-27 16:58:00| download_sheet|       English|   Chapter 2|     web|               5|
|   STU100099|2026-02-25 08:37:00|    video_watch|          Math|   Chapter 1|     ios|              43|
|   STU100238|2025-08-03 17:03:00|    quiz_submit|       Physics|   Chapter 2|  tablet|              80|
|   STU100020|2025-12-26 01:46:00|practice_finish|       Biology|   Chapter 1|  tablet|              12|
|   STU100398|   03/10/2026 23:55|    video_watch|Social Studies|       Intro|     ios|              70|
|   STU100281|2025-08-05 19:33:00| practice_start|          Thai|Practice Set|     ios|               4|
|   STU100001|2025-09-18 12:23:00|    quiz_submit|Socia

### Normalize event_time

In [26]:
bronze_learning_df = (
    bronze_learning_df
    .withColumn(
        "event_time",
        format_then_cast_timestamp(
            "event_time",
            slash_pattern="MM/dd/yyyy HH:mm"
        )
    )
    .filter(
        F.col("event_time").isNotNull()
    )
)

bronze_learning_df.show(50)

+------------+---------------+--------------+------------+--------+----------------+-------------------+
|student_code|     event_name|       subject|     chapter|platform|duration_minutes|         event_time|
+------------+---------------+--------------+------------+--------+----------------+-------------------+
|   STU100294| download_sheet|       English|   Chapter 2|     web|               5|2026-02-27 16:58:00|
|   STU100099|    video_watch|          Math|   Chapter 1|     ios|              43|2026-02-25 08:37:00|
|   STU100238|    quiz_submit|       Physics|   Chapter 2|  tablet|              80|2025-08-03 17:03:00|
|   STU100020|practice_finish|       Biology|   Chapter 1|  tablet|              12|2025-12-26 01:46:00|
|   STU100398|    video_watch|Social Studies|       Intro|     ios|              70|2026-10-03 23:55:00|
|   STU100281| practice_start|          Thai|Practice Set|     ios|               4|2025-08-05 19:33:00|
|   STU100001|    quiz_submit|Social Studies|   Chapter

### Normalize event_name

In [27]:
bronze_learning_df.select("event_name").distinct().show(40,truncate=False)

+---------------+
|event_name     |
+---------------+
|practice_start |
|quiz submit    |
|lesson open    |
|download_sheet |
|QUIZ_SUBMIT    |
|practice start |
|practice_finish|
|LESSON_OPEN    |
|practice finish|
|download sheet |
|PRACTICE_FINISH|
|video watch    |
|VIDEO_WATCH    |
|quiz_submit    |
|video_watch    |
|lesson_open    |
|DOWNLOAD_SHEET |
|PRACTICE_START |
+---------------+



In [28]:
bronze_learning_df = (
    bronze_learning_df
    .withColumn(
        "lower", F.lower(
            F.col("event_name")
        )
    ).withColumn(
        "clean", F.regexp_replace(
            F.lower(F.col("lower")), r"\s+", "_")
    ).drop(
        "event_name",
        "lower"
    ).withColumn(
        "event_name", F.col("clean")
    ).drop(
        "clean"
    )
)

bronze_learning_df.show(truncate=False)

+------------+--------------+------------+--------+----------------+-------------------+---------------+
|student_code|subject       |chapter     |platform|duration_minutes|event_time         |event_name     |
+------------+--------------+------------+--------+----------------+-------------------+---------------+
|STU100294   |English       |Chapter 2   |web     |5               |2026-02-27 16:58:00|download_sheet |
|STU100099   |Math          |Chapter 1   |ios     |43              |2026-02-25 08:37:00|video_watch    |
|STU100238   |Physics       |Chapter 2   |tablet  |80              |2025-08-03 17:03:00|quiz_submit    |
|STU100020   |Biology       |Chapter 1   |tablet  |12              |2025-12-26 01:46:00|practice_finish|
|STU100398   |Social Studies|Intro       |ios     |70              |2026-10-03 23:55:00|video_watch    |
|STU100281   |Thai          |Practice Set|ios     |4               |2025-08-05 19:33:00|practice_start |
|STU100001   |Social Studies|Chapter 1   |web     |81  

In [29]:
bronze_learning_df.select("event_name").distinct().show(40,truncate=False)

+---------------+
|event_name     |
+---------------+
|practice_start |
|download_sheet |
|practice_finish|
|quiz_submit    |
|video_watch    |
|lesson_open    |
+---------------+



### Silver Schema

In [30]:
silver_learning_df = bronze_learning_df.select(
    "student_code",
    "event_time",
    "event_name",
    "subject",
    "chapter",
    "duration_minutes",
    "platform"
)

print(silver_learning_df.count())
silver_learning_df.printSchema()

929
root
 |-- student_code: string (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- event_name: string (nullable = true)
 |-- subject: string (nullable = true)
 |-- chapter: string (nullable = true)
 |-- duration_minutes: integer (nullable = true)
 |-- platform: string (nullable = true)



In [44]:
silver_learning_df.show(1000,truncate=False)

+------------+-------------------+---------------+--------------+------------+----------------+--------+
|student_code|event_time         |event_name     |subject       |chapter     |duration_minutes|platform|
+------------+-------------------+---------------+--------------+------------+----------------+--------+
|STU100294   |2026-02-27 16:58:00|download_sheet |English       |Chapter 2   |5               |web     |
|STU100099   |2026-02-25 08:37:00|video_watch    |Math          |Chapter 1   |43              |ios     |
|STU100238   |2025-08-03 17:03:00|quiz_submit    |Physics       |Chapter 2   |80              |tablet  |
|STU100020   |2025-12-26 01:46:00|practice_finish|Biology       |Chapter 1   |12              |tablet  |
|STU100398   |2026-10-03 23:55:00|video_watch    |Social Studies|Intro       |70              |ios     |
|STU100281   |2025-08-05 19:33:00|practice_start |Thai          |Practice Set|4               |ios     |
|STU100001   |2025-09-18 12:23:00|quiz_submit    |Socia

## Sales

In [53]:
bronze_sales_df = sales_df

print(f"Records = {bronze_sales_df.count()}")
bronze_sales_df.show()

Records = 1000
+------------+------------+-------------------+--------------------+---------------+-------+--------------+
|student_code|    order_id|         order_date|        product_name|   product_type| amount|payment_status|
+------------+------------+-------------------+--------------------+---------------+-------+--------------+
|   STU100334|ORD202500000|2025-10-19 03:06:00|Chemistry Crash C...|recorded_course|   4990|      Complete|
|   STU100407|ORD202500000|2025-10-01 12:15:00|Math Intensive Bo...| mock_exam_pack|    990|     completed|
|   STU100286|ORD202500002|2025-10-28 07:58:00|Chemistry Crash C...|   trial_course|   0.99|        failed|
|   STU100103|ORD202500003|2025-12-10 14:14:00|M6 Final Review B...|recorded_course|   1990|     completed|
|   STU100003|ORD202500002|2026-02-02 00:35:00|English Grammar A...|    live_course|   4990|      Complete|
|   STU100366|ORD202500005|2025-12-04 22:27:00|   Biology Exam Prep|         bundle|2490.99|       pending|
|   STU100169

### Trim student_code and order_id

In [54]:
bronze_sales_df = (
    bronze_sales_df
    .withColumn(
        "student_code", F.trim(F.col("student_code"))
    )
    .withColumn(
        "order_id", F.trim(F.col("order_id"))
    )
)

bronze_sales_df.show(truncate=False)

+------------+------------+-------------------+---------------------------+---------------+-------+--------------+
|student_code|order_id    |order_date         |product_name               |product_type   |amount |payment_status|
+------------+------------+-------------------+---------------------------+---------------+-------+--------------+
|STU100334   |ORD202500000|2025-10-19 03:06:00|Chemistry Crash Course     |recorded_course|4990   |Complete      |
|STU100407   |ORD202500000|2025-10-01 12:15:00|Math Intensive Bootcamp    |mock_exam_pack |990    |completed     |
|STU100286   |ORD202500002|2025-10-28 07:58:00|Chemistry Crash Course     |trial_course   |0.99   |failed        |
|STU100103   |ORD202500003|2025-12-10 14:14:00|M6 Final Review Bundle     |recorded_course|1990   |completed     |
|STU100003   |ORD202500002|2026-02-02 00:35:00|English Grammar Accelerator|live_course    |4990   |Complete      |
|STU100366   |ORD202500005|2025-12-04 22:27:00|Biology Exam Prep          |bundl

### Drop NULL on student_code and order_id

In [55]:
bronze_sales_df = (
    bronze_sales_df
    .filter(
        (F.col("student_code").isNotNull()) &
        (F.col("student_code").rlike(r"^STU\d{6}$")) &
        (F.col("order_id").isNotNull()) &
        (F.col("order_id").rlike(r"^ORD\d{9}$"))
    )
)

bronze_sales_df.show(1000)

+------------+------------+-------------------+--------------------+---------------+-------+--------------+
|student_code|    order_id|         order_date|        product_name|   product_type| amount|payment_status|
+------------+------------+-------------------+--------------------+---------------+-------+--------------+
|   STU100334|ORD202500000|2025-10-19 03:06:00|Chemistry Crash C...|recorded_course|   4990|      Complete|
|   STU100407|ORD202500000|2025-10-01 12:15:00|Math Intensive Bo...| mock_exam_pack|    990|     completed|
|   STU100286|ORD202500002|2025-10-28 07:58:00|Chemistry Crash C...|   trial_course|   0.99|        failed|
|   STU100103|ORD202500003|2025-12-10 14:14:00|M6 Final Review B...|recorded_course|   1990|     completed|
|   STU100003|ORD202500002|2026-02-02 00:35:00|English Grammar A...|    live_course|   4990|      Complete|
|   STU100366|ORD202500005|2025-12-04 22:27:00|   Biology Exam Prep|         bundle|2490.99|       pending|
|   STU100169|ORD202500006|2

### Cast amount into decimal with 2 places

In [56]:
bronze_sales_df = bronze_sales_df.withColumn(
            "amount_int", F.col("amount").cast("decimal(12,2)")
        ).drop(
            "amount"
        ).withColumn(
            "amount", F.col("amount_int")
        ).drop(
            "amount_int"
        )

bronze_sales_df.printSchema()

root
 |-- student_code: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)



In [57]:
bronze_sales_df = (
    bronze_sales_df.withColumn(
        "formatted", F.to_timestamp(
            F.col("order_date")
        )
    )
    .drop(
        "order_date"
    )
    .withColumn(
        "order_date", F.col("formatted")
    )
    .drop(
        "formatted"
    )
)

bronze_sales_df.printSchema()

root
 |-- student_code: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- order_date: timestamp (nullable = true)



### Fix trial course's price

In [58]:
bronze_sales_df = (
    bronze_sales_df
    .withColumn(
        "amount", F.when(
            F.col("product_type") == F.lit("trial_course"), F.lit(0)
        ).otherwise(
            F.col("amount")
        )
    )
)

### Normalize payment_status and keep only success transaction

In [59]:
bronze_sales_df.select("payment_status").distinct().show(100)

+--------------+
|payment_status|
+--------------+
|     completed|
|       success|
|      Complete|
|        failed|
|          PAID|
|          paid|
|       pending|
+--------------+



In [60]:
bronze_sales_df.withColumn(
        "lower", F.lower(F.col("payment_status"))
    ).select("lower").distinct().show(100)

+---------+
|    lower|
+---------+
|completed|
|  success|
|   failed|
|     paid|
|  pending|
| complete|
+---------+



In [61]:
bronze_sales_df = (
    bronze_sales_df
    .withColumn(
        "lower", F.lower(F.col("payment_status"))
    )
    .filter(
        F.col("payment_status").isin("success", "completed", "paid", "complete")
    )
    .withColumn(
        "payment_status", F.lit("success")
    )
    .drop(
        "lower"
    )
)

bronze_sales_df.select("payment_status").distinct().show()

+--------------+
|payment_status|
+--------------+
|       success|
+--------------+



In [62]:
bronze_sales_df.show(1000)

+------------+------------+--------------------+---------------+--------------+-------+-------------------+
|student_code|    order_id|        product_name|   product_type|payment_status| amount|         order_date|
+------------+------------+--------------------+---------------+--------------+-------+-------------------+
|   STU100407|ORD202500000|Math Intensive Bo...| mock_exam_pack|       success| 990.00|2025-10-01 12:15:00|
|   STU100103|ORD202500003|M6 Final Review B...|recorded_course|       success|1990.00|2025-12-10 14:14:00|
|   STU100169|ORD202500006| TCAS Mock Exam Pack|   trial_course|       success|   0.00|2025-12-29 00:01:00|
|   STU100066|ORD202500008|    Math Trial Class|   trial_course|       success|   0.00|2025-07-10 04:47:00|
|   STU100285|ORD202500011|   Biology Exam Prep|    live_course|       success| 490.00|2025-12-21 09:00:00|
|   STU100087|ORD202500014|Physics Formula M...|   trial_course|       success|   0.00|2025-11-28 04:18:00|
|   STU100412|ORD202500015|P

### Silver Schema

In [63]:
silver_sales_df = bronze_sales_df.select(
        "student_code",
        "order_id",
        "order_date",
        "product_name",
        "product_type",
        "amount",
        "payment_status"
    )

print(silver_sales_df.count())
silver_sales_df.printSchema()

426
root
 |-- student_code: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_type: string (nullable = true)
 |-- amount: decimal(12,2) (nullable = true)
 |-- payment_status: string (nullable = false)



In [64]:
silver_sales_df.show(1000,truncate=False)

+------------+------------+-------------------+---------------------------+---------------+-------+--------------+
|student_code|order_id    |order_date         |product_name               |product_type   |amount |payment_status|
+------------+------------+-------------------+---------------------------+---------------+-------+--------------+
|STU100407   |ORD202500000|2025-10-01 12:15:00|Math Intensive Bootcamp    |mock_exam_pack |990.00 |success       |
|STU100103   |ORD202500003|2025-12-10 14:14:00|M6 Final Review Bundle     |recorded_course|1990.00|success       |
|STU100169   |ORD202500006|2025-12-29 00:01:00|TCAS Mock Exam Pack        |trial_course   |0.00   |success       |
|STU100066   |ORD202500008|2025-07-10 04:47:00|Math Trial Class           |trial_course   |0.00   |success       |
|STU100285   |ORD202500011|2025-12-21 09:00:00|Biology Exam Prep          |live_course    |490.00 |success       |
|STU100087   |ORD202500014|2025-11-28 04:18:00|Physics Formula Mastery    |trial